In [ ]:
# Project features
# 1. Upload one PDF or DOCX document.
# 2. Extract document text automatically.
# 3. Split text into smaller chunks.
# 4. Generate text embeddings.
# 5. Store embeddings in FAISS.
# 6. Retrieve relevant information based on questions.
# 7. Generate answers using Gemini.
# 8. Display source page references.
# 9. Ask unlimited questions.
# 10. Type bye to terminate the chatbot.

# flow will be :Sentence Transformer → FAISS → Gemini → AI Answer

In [ ]:
# 1. important imports

!pip install -q google-genai sentence-transformers faiss-cpu pypdf python-docx


In [ ]:
# !pip install =	Installs Python libraries in Google Colab.
# -q	 = Quiet mode; reduces installation messages.
# google-genai	= Connects our Python project to Google's Gemini AI.
# sentence-transformers	= Converts text into numerical embeddings.
# faiss-cpu	 = Searches for similar embeddings quickly.
# pypdf	= Extracts text from PDF files.
# python-docx	= Extracts text from Word .docx files.

In [ ]:
import faiss
import numpy as np

from sentence_transformers import SentenceTransformer
from google import genai

from google.colab import files
from pypdf import PdfReader
from docx import Document

import getpass

In [ ]:
# Library	Main job
# Sentence Transformers	Understand text similarity
# FAISS	Find relevant text
# Gemini	Generate answers
# PyPDF	Read PDF
# python-docx	Read Word
# getpass = allows to type api key without entirely displaying it

In [ ]:
# API_KEY = getpass.getpass("Enter Gemini API Key: ")

# client = genai.Client(api_key="AQ.Ab8RN6JOiVna00kDy*********************")

embedder = SentenceTransformer("all-MiniLM-L6-v2")

print("AI models loaded successfully!")
# commented my api key - confidentiality purpose

In [ ]:
# genai.Client() creates a connection object for Gemini.
# api_key=API_KEY provides the key needed for authentication.

In [ ]:
# -> SentenceTransformer() loads a pretrained embedding model.
# -> "all-MiniLM-L6-v2" is the model name.
# -> embedder stores the loaded model.
# What does this model do?

# Suppose we have these sentences:
# -> "I love Python programming."
# -> "Python is my favourite programming language."
# Although the words are different, their meanings are similar.

# The embedding model converts these sentences into numerical vectors that can be compared for similarity.

In [ ]:
# Sentence Transformers creates embeddings. Gemini generates answers. They are two different models with two different jobs.

In [ ]:
uploaded = files.upload()

file_path = list(uploaded.keys())[0]

sections = [] #need somewhere to store the extracted document text and its source information.

if file_path.lower().endswith(".pdf"):

    reader = PdfReader(file_path)

    for page_no, page in enumerate(reader.pages, start=1):

        text = page.extract_text() or ""

        if text.strip():
            sections.append((text, f"Page {page_no}"))

elif file_path.lower().endswith(".docx"):

    doc = Document(file_path)

    text = "\n".join(p.text for p in doc.paragraphs if p.text.strip())

    if text.strip():
        sections.append((text, "Word Document"))

else:
    raise ValueError("Upload only PDF or DOCX files.")

if not sections:
    raise ValueError("No readable text found in document.")

print("Document uploaded successfully!", file_path, len(sections))

In [ ]:
# uploaded.keys() gets the uploaded filenames.

# list() converts them into a list.

# [0] selects the first filename.

# lower() converts the filename into lowercase.

# endswith(".pdf") checks whether the filename ends with .pdf.

# if executes the PDF-reading code when the condition is true.

In [ ]:
# -> reader.pages contains all PDF pages.
# -> enumerate() gives us both the page number and page object.
# -> start=1 starts counting from page 1.
# -> page_no stores the page number.
# -> page represents the current page.
# For example, if the PDF has 5 pages, this loop runs 5 times.

# append() adds a new item to the list.

# (text, f"Page {page_no}") stores the extracted text and its page number together.

In [ ]:
# doc.paragraphs gets all paragraphs.

# p.text gets the text of each paragraph.

# if p.text.strip() ignores empty paragraphs.

# "\n".join() combines all paragraph texts with a new line between them.

In [ ]:
# Imagine a 100-page employee handbook.

# If we send the entire document to Gemini for every question, it can be inefficient and may exceed the model's context limits.

# So, we divide the document into smaller pieces called chunks.

In [ ]:
chunks = []
sources = []

chunk_size = 500
overlap = 100  #allow 100 characters to overlap between consecutive chunks.
# Each chunk can contain up to 500 characters.
for text, source in sections:

    for i in range(0, len(text), chunk_size - overlap): #chunk_size - overlap calculates the step size. eg: 500 - 100 = 400
#So the next chunk starts 400 characters after the previous chunk.
        chunk = text[i:i + chunk_size]

        if chunk.strip():
            chunks.append(chunk)
            sources.append(source)

if not chunks:
    raise ValueError("No document chunks were created.")

print("Total chunks:", len(chunks))#eg:  Total chunks: 25 then It means our document has been divided into 25 smaller text pieces.

In [ ]:
embeddings = embedder.encode(chunks)

embeddings = np.array(embeddings, dtype=np.float32)

print("Embeddings generated successfully!", embeddings.shape)

# Embeddings are numerical representations of text that help computers compare meanings.

In [ ]:
#  An embedding model converts text into numbers so that similar meanings can be searched mathematically.

# -> embedder is our Sentence Transformer model from Step 3.
# -> encode() converts text into numerical vectors.
# -> chunks contains all our document chunks.
# -> embeddings stores the resulting numerical representations.
# For example, conceptually:
# "Python is easy to learn."
#         ↓
# [0.23, 0.81, 0.12, 0.45, ...]

In [ ]:
# FAISS helps us find the most similar numerical vectors quickly.
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("FAISS database created!", index.ntotal)# index.ntotal gives the number of vectors stored in FAISS.

In [ ]:
# -> faiss accesses the FAISS library.
# -> IndexFlatL2() creates an index using L2 (Euclidean) distance.
# -> dimension tells FAISS the number of values in each vector.
# -> index stores our searchable index.
# In simple words, FAISS prepares a place where we can search for similar embeddings.

In [ ]:
# This inserts all document embeddings into the FAISS index.

# Now FAISS can search through them.


In [ ]:

print("\n====================================")
print("          SREE'S CHAT AI")
print("====================================")
print("Chat with your uploaded document!")
print("Type 'bye' to exit.")

while True:

    # Get user question
    question = input("\nYou: ")

    # Exit chatbot
    if question.lower().strip() == "bye":

        print("Sree's AI: Thank you! bye! Have a best day....")
        break

    # Check empty question
    if not question.strip():

        print("Sree's AI: Please enter a question.")
        continue

    # Convert question into embedding
    q_embedding = embedder.encode([question])

    q_embedding = np.array(
        q_embedding, dtype=np.float32
    ) #Converts the question embedding into the numerical format FAISS expects.

    # Search relevant chunks
    k = min(3, len(chunks))

    distances, indices = index.search(
        q_embedding, k
    )

    # Collect retrieved context
    retrieved_text = ""
    retrieved_sources = []

    for idx in indices[0]:

        retrieved_text += chunks[idx] + "\n"

        source = sources[idx]

        if source not in retrieved_sources:
            retrieved_sources.append(source)

    # Prepare Gemini prompt
    prompt = f"""
    You are Sree's AI, an intelligent document assistant.

    Answer the question using ONLY the given context.

    If the answer is not available in the context,
    respond:
    Information not found in the document.

    Explain in simple and clear language.

    Context:
    {retrieved_text}

    Question:
    {question}

    Answer:
    """

    # Generate AI answer
    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt
    )

    # Display answer
    print("\nDocuChat AI:", response.text)

    # Display source references
    print("\nSources:", ", ".join(retrieved_sources))
# note: FAISS does not generate the answer. It only finds relevant document chunks.


          DOCUCHAT AI
Chat with your uploaded document!
Type 'bye' to exit.

DocuChat AI: Based on the provided document, there are 35 numbered topics listed.

Sources: Page 1, Page 2

DocuChat AI: The course duration is 80 hours.

Sources: Page 1, Page 2

DocuChat AI: The topics related to JSP covered in the document are:

*   **Introduction:** How to create your first JSP.
*   **Components:** Scriplets, expressions, and declarations.
*   **Features:** Implicit objects, action elements, and session handling.
*   **Utilities:** JSP-Auto refresh.
*   **Architecture:** MVC architecture.
*   **Practical application:** HTML forms and a project involving JSP.

Sources: Page 1, Page 2


In [ ]:
# indices contains the positions of the matching chunks.

# indices[0] gets the results for our single question.

# idx represents one matching chunk's position.

In [ ]:
# Retrieval	FAISS finds relevant document chunks.
# Augmentation	Adds retrieved chunks to Gemini's prompt.
# Generation	Gemini generates the answer.
# That is RAG